In [1]:
import pandas as pd
from sklearn.datasets import load_iris
from google.cloud import bigquery
from google.oauth2 import service_account

In [2]:
# Cargar el conjunto de datos Iris desde sklearn
iris = load_iris()
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
iris_df['target'] = iris.target

In [3]:
# Convertir los nombres de las columnas a un formato adecuado (sin espacios)
iris_df.columns = [col.replace(" ", "_").replace("(", "").replace(")", "") for col in iris_df.columns]

In [4]:
iris_df

,sepal_length_cm,sepal_width_cm,petal_length_cm,petal_width_cm,target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [5]:
iris_df['target'].value_counts()

target
0    50
1    50
2    50
Name: count, dtype: int64

In [8]:
client = bigquery.Client()

In [9]:
project_id = client.project
project_id

'capable-hash-432501-a6'

In [10]:
# Definir el nombre del dataset y tabla en BigQuery
dataset_id = 'iris_test'  # Cambia esto por el ID de tu dataset en BigQuery
table_id = 'iris_data'  # El nombre de la tabla a la que subirás los datos

In [17]:
# Construir el ID completo de la tabla (incluye proyecto, dataset y tabla)
table_full_id = f"{project_id}.{dataset_id}.{table_id}"

In [19]:
# Verificar si el dataset ya existe
try:
    client.get_dataset(dataset_full_id)  # Este método lanza una excepción si el dataset no existe
    print(f"El dataset {dataset_id} ya existe.")
except:
    print(f"El dataset {dataset_id} no existe. Creándolo...")
    # Crear el dataset si no existe
    dataset = bigquery.Dataset(dataset_full_id)
    dataset.location = "US"  # Cambia la ubicación si es necesario
    dataset = client.create_dataset(dataset)  # Esto creará el dataset
    print(f"Dataset {dataset_id} creado exitosamente.")


El dataset iris_test ya existe.


In [14]:
# Construir el ID completo del dataset
dataset_full_id = f"{project_id}.{dataset_id}"

In [18]:
# Subir los datos a BigQuery
job = client.load_table_from_dataframe(iris_df, table_full_id)

In [20]:
# Esperar a que termine la carga
job.result()

LoadJob<project=capable-hash-432501-a6, location=US, id=1184cc36-7b94-4746-b23a-521d9c60080a>

In [21]:
# Verificar que los datos se subieron correctamente
table = client.get_table(table_full_id)
print(f"Se subieron {table.num_rows} filas a la tabla {table_full_id}.")

Se subieron 300 filas a la tabla capable-hash-432501-a6.iris_test.iris_data.
